In [2]:
#Import
!pip install linearmodels
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from linearmodels.iv import IV2SLS

  Using cached linearmodels-7.0-cp313-cp313-win_amd64.whl.metadata (10 kB)
  Using cached pyhdfe-0.2.0-py3-none-any.whl.metadata (4.0 kB)
  Using cached formulaic-1.2.1-py3-none-any.whl.metadata (7.0 kB)
  Using cached interface_meta-1.3.0-py3-none-any.whl.metadata (6.7 kB)
Using cached linearmodels-7.0-cp313-cp313-win_amd64.whl (1.5 MB)
Using cached formulaic-1.2.1-py3-none-any.whl (117 kB)
Using cached interface_meta-1.3.0-py3-none-any.whl (14 kB)
Using cached pyhdfe-0.2.0-py3-none-any.whl (19 kB)

   -------------------- ------------------- 2/4 [formulaic]
   -------------------- ------------------- 2/4 [formulaic]
   ------------------------------ --------- 3/4 [linearmodels]
   ------------------------------ --------- 3/4 [linearmodels]
   ------------------------------ --------- 3/4 [linearmodels]
   ------------------------------ --------- 3/4 [linearmodels]
   ------------------------------ --------- 3/4 [linearmodels]
   ------------------------------ --------- 3/4 [linearmode


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
# Load master panel
dataset = pd.read_csv(r'C:\Users\Spandan Dutta\econometrics_project\data\processed\master_panel.csv',parse_dates=['date'],index_col='date')
dataset.head()

,yield_1y,yield_2y,yield_3y,yield_4y,yield_5y,yield_6y,yield_7y,yield_8y,yield_9y,yield_10y,...,d_yield_30y,d_slope_10y_2y,d_fpi,d_us_10y,d_vix,d_brent,d_dxy,d_cpi_inflation,d_laf_net,d_repo_rate
date,,,,,,,,,,,,,,,,,,,,,
2021-04-30,3.80380,4.25790,4.89680,5.39230,5.68390,6.03340,6.21780,6.36770,6.13840,6.35670,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2021-05-31,3.90921,4.36907,4.85644,5.26522,5.65988,6.08705,6.24015,6.51243,6.20216,6.25843,...,0.25824,-0.20944,-0.028150,-0.07,2.344310,3.727711,-1.3263,NaN,81460.818925,0.0
2021-06-30,4.05553,4.57015,5.12149,5.61436,5.80350,6.16466,6.34695,6.45004,6.29422,6.39280,...,0.08284,-0.06671,-0.028150,-0.13,-2.803682,4.629880,1.7957,NaN,-50881.678925,0.0
2021-07-31,3.81018,4.34746,4.94329,5.48950,5.83545,6.15497,6.38605,6.44795,6.37177,6.22949,...,0.06602,0.05938,0.005549,-0.21,0.646515,2.001818,0.1181,NaN,-56849.254624,0.0
2021-08-31,3.68540,4.22070,4.79860,5.37100,5.75670,6.01390,6.18990,6.25490,6.23740,6.37140,...,-0.07530,0.26867,0.005549,0.06,-0.130606,-4.418766,0.4362,NaN,-74986.290323,0.0


In [11]:
#regression 1
# Variables to be differenced
diff_cols = ['yield_2y', 'yield_5y', 'yield_10y', 'yield_30y','slope_10y_2y', 'slope_30y_10y','us_10y', 'vix', 'dxy', 'brent', 'laf_net']
#differencing
for col in diff_cols:
    dataset[f'd_{col}'] = dataset[col].diff()

# Controls 
controls = ['d_us_10y', 'd_vix', 'd_dxy', 'd_brent','repo_rate', 'cpi_inflation', 'd_laf_net']

# Regression dataset
reg_vars = ['d_slope_10y_2y','d_yield_10y','fpi','jpmorgan_weight'] + controls
df_reg = dataset[reg_vars].dropna()
print(df_reg.shape)

#OLS
Y=df_reg['d_slope_10y_2y']
X=sm.add_constant(df_reg[['fpi'] + controls])
model1 = sm.OLS(Y,X).fit()
model1.summary()

(39, 11)


<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:         d_slope_10y_2y   R-squared:                       0.199
Model:                            OLS   Adj. R-squared:                 -0.014
Method:                 Least Squares   F-statistic:                    0.9333
Date:                Fri, 27 Mar 2026   Prob (F-statistic):              0.504
Time:                        04:12:17   Log-Likelihood:                 23.503
No. Observations:                  39   AIC:                            -29.01
Df Residuals:                      30   BIC:                            -14.03
Df Model:                           8                                         
Covariance Type:            nonrobust                                         
=================================================================================
                    coef    std err          t      P>|t|      [0.025      0.975]
---------------------------------------------------------------------------------
const             0.2221      0.357      0.622      0.539      -0.507       0.951
fpi              -0.0632      0.077     -0.815      0.421      -0.221       0.095
d_us_10y         -0.0850      0.122     -0.699      0.490      -0.334       0.164
d_vix            -0.0013      0.010     -0.132      0.896      -0.021       0.019
d_dxy             0.0084      0.018      0.458      0.650      -0.029       0.046
d_brent           0.0017      0.004      0.450      0.656      -0.006       0.009
repo_rate         0.0260      0.034      0.775      0.445      -0.043       0.095
cpi_inflation    -0.0585      0.035     -1.686      0.102      -0.129       0.012
d_laf_net      3.689e-07   6.87e-07      0.537      0.595   -1.03e-06    1.77e-06
==============================================================================
Omnibus:                       21.760   Durbin-Watson:                   1.998
Prob(Omnibus):                  0.000   Jarque-Bera (JB):               49.642
Skew:                          -1.277   Prob(JB):                     1.66e-11
Kurtosis:                       7.901   Cond. No.                     5.69e+05
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
[2] The condition number is large, 5.69e+05. This might indicate that there are
strong multicollinearity or other numerical problems.
"""